In [2]:
"""
%pip install -qU pypdf langchain_community
%pip install langchain_chroma langchain_openai
"""

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [21]:
import pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.ollama import OllamaEmbeddings
from langchain.vectorstores.chroma import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama

In [12]:
loader = PyPDFLoader("../pdfs/DDDLaravel.pdf")
docs = loader.load()

In [17]:
print(len(docs))

49


In [18]:
print(docs[0].page_content[0:100])
print(docs[0].metadata)

DOMAIN-DRIVEN
DESIGN WITH
LARAVEL
MARTIN JOO
The only design approach you need
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-04-16T13:55:08+00:00', 'source': '../pdfs/DDDLaravel.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1'}


In [14]:
embed = OllamaEmbeddings(model="nomic-embed-text")

In [16]:
# Before running, start ollama server
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=embed)

retriever = vectorstore.as_retriever()

In [22]:
llm=Ollama(model="mistral")

/tmp/ipykernel_216/960184826.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm=Ollama(model="mistral")


In [20]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [23]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [27]:
results = rag_chain.invoke({"input": "Please write to me a value object php class based on the DDD context you have"})

In [28]:
results['answer']

" Based on the provided context, here's an example of a PHP Value Object class in Domain-Driven Design (DDD) style:\n\n```php\nclass EmailValueObject\n{\n    private string $email;\n\n    public function __construct(string $email)\n    {\n        $this->email = $email;\n    }\n\n    public function value(): string\n    {\n        return $this->email;\n    }\n}\n```\n\nThis class represents a value object called `EmailValueObject`. It takes an email as its constructor argument, and the `value()` method returns the email. Note that a value object never has an ID in DDD. You can create multiple instances with the same email value, and they will be considered as the same logical entity."